## 1. Preprocessing
This section provides an initial examination of the dataset compiled by Mesquita and Pires (2024)[1]. According to their methodology, citations are detected by scanning the resolution code within the full text of General Assembly resolutions.

The first dataset, _ga_resolutions_1946_2019_, contains the resolution text, approval date, resolution code, and additional metadata.

After applying the citation‑detection procedure, the resulting dataset is _ga_citations_1946_2019_, which includes one row per citation edge. For example, if resolutions A, B, and C cite resolution X, the dataset will contain one row for each of A, B, and C, each having metadata of X and the respectively A, B or C.

The dataset also includes a field named topic, which appears to correspond to the resolution’s title. In many cases this field does not match the actual title, although for the most frequently cited resolutions it aligns reasonably well. This inconsistency requires correction. In the following code chunks, the data is loaded and the top 30 most cited resolutions are displayed.

Subsequently, we rescan the titles and correct those that appear to be inaccurately recorded. We also identify false positives among the most cited resolutions and collect cases where a citation should exist but is missing from the dataset.

Finally, we recombine all corrected elements to produce a clean and consistent database.

[1] Rafael Mesquita, Antonio Pires, The references of the nations: Introducing a corpus of United Nations General Assembly resolutions since 1946 and their citation network, Journal of Peace Research, Volume 62, Issue 4, July 2025, Pages 1279–1291, https://doi.org/10.1177/00223433241254997

In [36]:
import os
import pandas as pd
import re

# ----------------------------
# SETTINGS
# ----------------------------
CITATIONS_CSV = "./ga_citations_1946_2019.csv"
RESOLUTIONS_CSV = "./ga_resolutions_1946_2019.csv"

# ----------------------------
# LOAD DATA
# ----------------------------
if not os.path.isfile(CITATIONS_CSV):
    print(f"Error: File '{CITATIONS_CSV}' does not exist.")
else:
    print("File found. Loading...")

    try:
        # Load in chunks to avoid memory issues
        chunksize = 200000
        chunks = []

        for chunk in pd.read_csv(CITATIONS_CSV, chunksize=chunksize):
            chunks.append(chunk)

        df1 = pd.concat(chunks, ignore_index=True)
        print("Citations file loaded successfully.")

    except Exception as e:
        print(f"Error loading CSV file: {e}")

    finally:
        pass

if not os.path.isfile(RESOLUTIONS_CSV):
    print(f"Error: File '{RESOLUTIONS_CSV}' does not exist.")
else:
    print("File found. Loading...")

    try:
        # Load in chunks to avoid memory issues
        chunksize = 200000
        chunks = []

        for chunk in pd.read_csv(RESOLUTIONS_CSV, chunksize=chunksize):
            chunks.append(chunk)

        df2 = pd.concat(chunks, ignore_index=True)
        print("CSV file loaded successfully.")

    except Exception as e:
        print(f"Error loading CSV file: {e}")

    finally:
        pass

File found. Loading...
Citations file loaded successfully.
File found. Loading...
CSV file loaded successfully.


### First sanity check

In the following step, we verify that the merged dataset ga_citations_1946_2019 is fully consistent with the original ga_resolutions_1946_2019. After combining both sources, the content associated with each resolution should match exactly, ensuring that no information is lost or altered during the integration process.


In [2]:
print('Columns in df1. The most detailed dataframe, with citated and citators:')
print(df1.columns)
print('\nColumns in df2. The dataframe with the resolutions only:')
print(df2.columns)

#are content_giv equal to content? exactly the same?

df = pd.merge(df1, df2, left_on='res_id2_doc_giving_cite', right_on='res_id2', how='inner')
df['content_giv_equal_content'] = df['content_giv'] == df['content']
print('\nNumber of rows in df1: ', len(df1))
print('Number of rows in df2: ', len(df2))

print('Number of rows in df1 when merged with df2 on res_id2: ', len(df))

print('\n\nFirst sanity check, is the resolution transcript the same given a resolution in both dataframe?')
print('Number of rows where content_giv == content: ', df['content_giv_equal_content'].sum())

Columns in df1. The most detailed dataframe, with citated and citators:
Index(['res_id2_doc_giving_cite', 'res_id3_giv', 'res_id2_unlet_giv',
       'alt_id_dic_giv', 'session_type_giv', 'session_reg_giv',
       'session_sp_giv', 'session_es_giv', 'resn_giv', 'res_letter_giv',
       'date_p_giv', 'date_c_giv', 'location_giv', 'topic_giv', 'filename_giv',
       'content_giv', 'cited_res', 'cited_res_conv', 'n_outg_cit',
       'res_id2_doc_receiv_cite', 'res_id3_recv', 'res_id2_unlet_recv',
       'alt_id_dic_recv', 'session_type_recv', 'session_reg_recv',
       'session_sp_recv', 'session_es_recv', 'resn_recv', 'res_letter_recv',
       'date_p_recv', 'date_c_recv', 'location_recv', 'topic_recv',
       'filename_recv', 'content_recv'],
      dtype='object')

Columns in df2. The dataframe with the resolutions only:
Index(['res_id2', 'res_id3', 'res_id2_unlet', 'alt_id_dic', 'session_type',
       'session_reg', 'session_sp', 'session_es', 'resn', 'res_letter',
       'date_p', 'dat

### New discovered nodes
Along this analysis we will inspect and find new references that we should take into consideration for a further LLM finetune. We can already start by creating a DataFrame to collect this new edges.

In [3]:
new_cites = pd.DataFrame({
    "res_id2_doc_giving_cite": pd.Series(dtype=str),
    "res_id2_unlet_recv": pd.Series(dtype=str)
})

In [4]:
# Important IDs: 'res_id2_doc_receiv_cite', 'res_id3_recv'

print("Let's take a look of the top 30 most cited resolutions.\n")
pd.set_option('display.max_colwidth', None)
top_cited = (
    df1.groupby(['filename_recv', 'topic_recv', 'res_id2_doc_receiv_cite', 'res_id3_recv'])
      .size()
      .reset_index(name='citation_count')
      .sort_values(by='citation_count', ascending=False)
)

# Get top 30
top_30 = top_cited.head(30)

# Reorder columns
top_30 = top_30[['topic_recv', 'citation_count', 'res_id3_recv', 'res_id2_doc_receiv_cite']]

print(top_30['topic_recv'])
pd.reset_option('display.max_colwidth')
print('\n\n',top_30)

Let's take a look of the top 30 most cited resolutions.

3828                                                                                                                                                                      universal declaration of human rights
298                                                                                                                               declaration on the granting of independence to colonial countries and peoples
7629                                                                                                                                                                                     millennium declaration
60                                                                                                                                                                       establishment of tax equilization fund
12399                                                                    general principles to serve as guideli

## Remarked resolutions for further study:
- **establishment of tax equilization fund:** This is only the title of the part A of the resolution. The title of the resolution itself is **Use of income derived from the Staff Assessment Plan**.
It is needed to check if there are registers of citation based on the wrong title.

- **convention on the elimination of all formsof discrimination against women**:
There is a typo in the topic name. It is needed to check if the scan was based using the title with a typo.

- **treaty for the prohibition of nuclear weapons in latin america**: This is a wrong title. The correct title is: Treaty on the Non-Proliferation of
Nuclear Weapons.

- **scale of assessments for the apportionment of expenses of un peacekeeping operations**: This contains a short name **UN** instead of full name, **united nations**. It is needed

- **Cross-cutting issues, 60/266 of 30 june 2006**: This is an incompleted title. The correct title is: **Administrative and budgetary aspects of the financing of the United Nations peacekeeping operations: cross-cutting issues**. In addition, it has a repeated name, which means that titles are not unique. For instance, resolution 59/296 of 22 June 2005 has the same name. This makes the scan of title not a good method to detect citation. Therefore, scanning for the resolution code is a more secure way to infer citation.

- **un convention against transnational organized crime**: United Nations should appear in the title, not its abbreviation. It is needed to check if there are citation without containing **united nations**, due to redundancy.

- **administrative and budgetary aspects of the financing of the United Nations peace-keeping operations, 47/218 A of 23 december 1992**. Notice that the title doesn't have **cross-cutting issues**. Differently from other **administrative and budgetary aspects**, this resolution has **peace-keeping** with a dash.
Moreover, the resolution code should not containt the **A** section mark, since there is no different sections in the whole resolution. This should be corrected.

- **administrative and budgetary aspects of the financing of the United Nations peace-keeping operations, 44/192 b of 21 december 1989**. Same comments of 47/218 a of 23 december 1992. With a further observation that 44/192 specifies the B section. In the resolution there is no specific separate title for A,B or C, it is difficult to infer that all citations actually refer specifically to the B section. Is this given because the resolution scan actually detected the B section after the code? Should it be cited as the section A? or none?

- **cross-cutting issues, 64/269 of 24 june 2010**. This is a correct title. Oficially, the title does not include the **Adminstrative and budgetary aspects** part.

In [5]:
title = 'universal declaration of human rights'
res_id = '217 (iii)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

Resolution  217 (iii) . Title:  universal declaration of human rights
Not reporting  217 (iii) as cite 12995
However, containing the title 0
Lets take a look

 []


In [6]:
title = 'declaration on the granting of independence to colonial countries and peoples'
res_id = '1514 (xv)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

pd.reset_option('display.max_colwidth')
pd.reset_option('display.max_columns')

Resolution  1514 (xv) . Title:  declaration on the granting of independence to colonial countries and peoples
Not reporting  1514 (xv) as cite 12733
However, containing the title 0
Lets take a look

 []


In [7]:
title = 'millennium declaration'
res_id = '55/2'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'])

print('\n\nAfter a revision, all should cite the 55/2 millennium declaration.')
df_aux = df_aux[['res_id2_doc_giving_cite']].drop_duplicates()
df_aux['res_id2_unlet_recv'] = res_id
new_cites = pd.concat([new_cites, df_aux], ignore_index=True)
print('Added to new_cites')

Resolution  55/2 . Title:  millennium declaration
Not reporting  55/2 as cite 4801
However, containing the title 6
Lets take a look

 40936      58/15 of 3 december 2003
48987    62/208 of 19 december 2007
58858    67/146 of 20 december 2012
65280    69/315 of 1 september 2015
65398     70/1 of 25 september 2015
70124    71/256 of 23 december 2016
Name: res_id3_giv, dtype: object


After a revision, all should cite the 55/2 millennium declaration.
Added to new_cites


In [8]:
title = 'use of income derived from the staff assessment plan'
res_id = '973 (x)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())


####### Check if registers are given, scanning the wrong title
print('\n\nWe can also check the cases where there is a register of citation but only the wrong title is detected.')
wrong_title = 'establishment of tax equilization fund'
positives = df1[df1['res_id2_unlet_recv'] == res_id]
df_aux = positives[positives['content_giv'].str.contains(wrong_title, case=False)]
df_aux = df_aux[~df_aux['content_giv'].str.contains(title, case=False)]
print('We have a total number of cases: ', len(df_aux['res_id3_giv'].unique()))

Resolution  973 (x) . Title:  use of income derived from the staff assessment plan
Not reporting  973 (x) as cite 13411
However, containing the title 0
Lets take a look

 []


Now we can take a look of the cases where there is a register of citation but only the wrong title is detected.
We have a total number of cases:  0


In [9]:
title = 'general principles to serve as guidelines for the sharing of the costs of future peace-keeping operations involving heaby expenditures'
res_id = '1874 (s-iv)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

Resolution  1874 (s-iv) . Title:  general principles to serve as guidelines for the sharing of the costs of future peace-keeping operations involving heaby expenditures
Not reporting  1874 (s-iv) as cite 13022
However, containing the title 0
Lets take a look

 []


In [10]:
title = '2005 world summit outcome'
res_id = '60/1'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

Resolution  60/1 . Title:  2005 world summit outcome
Not reporting  60/1 as cite 3646
However, containing the title 0
Lets take a look

 []


In [11]:
title = 'convention on the rights of the child'
res_id = '44/25'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

Resolution  44/25 . Title:  convention on the rights of the child
Not reporting  44/25 as cite 8215
However, containing the title 0
Lets take a look

 []


In [12]:
title = 'international covenant on economic, social and cultural rights, international covenant on civil and political rights and optional protocol to the international covenant on civil and political rights'
#Here we also consider parts of the title. Given that the resolution is divided in 3 parts: 2 convenant and 1 optional protocol.
title_part1 = 'international covenant on economic, social and cultural rights'
title_part2 = 'international covenant on civil and political rights'
title_part3 = 'optional protocol to the international covenant on civil and political rights'

res_id = '2200 (xxi)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux0 = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
df_aux1 = df_aux[df_aux['content_giv'].str.contains(title_part1, case=False)]
df_aux2 = df_aux[df_aux['content_giv'].str.contains(title_part2, case=False)]
df_aux3 = df_aux[df_aux['content_giv'].str.contains(title_part3, case=False)]
df_aux = pd.concat([df_aux0, df_aux1, df_aux2, df_aux3])
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

print('\n\nAll of them should cite 2200 a (xxi).')

df_aux = df_aux[['res_id2_doc_giving_cite']].drop_duplicates()
df_aux['res_id2_unlet_recv'] = res_id
new_cites = pd.concat([new_cites, df_aux], ignore_index=True)
print('Added to new_cites')

Resolution  2200 (xxi) . Title:  international covenant on economic, social and cultural rights, international covenant on civil and political rights and optional protocol to the international covenant on civil and political rights
Not reporting  2200 (xxi) as cite 12942
However, containing the title 332
Lets take a look

 ['2449 (xxiii) of 19 december 1968' '3009 (xxvii) of 18 december 1972'
 '3448 (xxx) of 9 december 1975' '31/103 of 15 december 1976'
 '32/61 of 8 december 1977' '32/62 of 8 december 1977'
 '32/118 of 16 december 1977' '32/121 of 16 december 1977'
 '33/73 of 15 december 1978' '33/179 of 20 december 1978'
 '34/146 of 17 december 1979' '34/155 of 17 december 1979'
 '34/178 of 17 december 1979' '35/172 of 15 december 1980'
 '35/178 of 15 december 1980' '35/201 of 16 december 1980'
 '36/22 of 9 november 1981' '36/29 of 13 november 1981'
 '36/60 of 25 november 1981' '36/149 b of 16 december 1981'
 '36/155 of 16 december 1981' '37/49 of 3 december 1982'
 '37/94 b of 10 dece

In [14]:
print("Let's check the different sections of resolution '2200 (xxi)'.")
res_id = '2200 (xxi)'
print(df2[df2['res_id2_unlet'] == res_id]['res_id2'].unique())
print("The covenants and the optional protocol are declared in the A section. Rarely they will cite the B or C, which are small paragraphs.")

Let's check the different sections of resolution '2200 (xxi)'.
['2200 a (xxi)' '2200 b (xxi)' '2200 c (xxi)']
The covenants and the optional protocol are declared in the A section. Rarely they will cite the B or C, which are small paragraphs.


In [15]:
title = 'Treaty on the Non-Proliferation of Nuclear Weapons'.lower()
res_id = '2373 (xxii)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

Resolution  2373 (xxii) . Title:  treaty on the non-proliferation of nuclear weapons
Not reporting  2373 (xxii) as cite 12845
However, containing the title 0
Lets take a look

 []


In [17]:
title = 'financing of the united nations emergency force'
avoid_pattern = (
    r'financing of the united nations emergency force '
    r'and (?:of )?the united nations disengagement observer force'
)
res_id = '3101 (xxviii)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title:', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
contains_title = df_aux['content_giv'].str.contains(
    title,
    case=False,
    na=False
)

contains_avoid = df_aux['content_giv'].str.contains(
    avoid_pattern,
    case=False,
    regex=True,
    na=False
)

df_aux = df_aux[contains_title & ~contains_avoid]

print('However, containing the title', len(df_aux['res_id2_doc_giving_cite']))

print('Lets take a look\n\n', df_aux['res_id3_giv'])

print('After a manual inspection, 31/95 and 35/11 do not cite the 3101 (xxviii), they cite as well the disengagement Observer force.')

Resolution  3101 (xxviii) . Title: financing of the united nations emergency force
Not reporting  3101 (xxviii) as cite 12258
However, containing the title 2
Lets take a look

 5766    31/95 b of 14 december 1976
8740     35/11 a of 3 november 1980
Name: res_id3_giv, dtype: object
After a manual inspection, 31/95 and 35/11 do not cite the 3101 (xxviii), they cite as well the disengagement Observer force.


## Financing of the United Nations Emergency Force
Known as resolution 3101. But there are other resolutions with the same name.

3211 (XXIX). Financing of the United Nations
Emergency Force and of the United Nations
Disengagement Observer Force

Which means that we cannot scann simply the title. Because 3211 containst the 3101 title.

- 3211 (XXIX): Financing of the United Nations
Emergency Force and of the United Nations
Disengagement Observer Force
- 3374 (XXX): Financing of the United Nations
Emergency Force and of the United Nations
Disengagement Observer Force

Therefore we need to avoid this pattern. This is already solved in the code above. The only two cases are 31/95, that uses a slightly different phrasing, and 35/11 which contains a typo in the transcription.

In [18]:
title = 'charter of economic rights and duties of states'
res_id = '3281 (xxix)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

Resolution  3281 (xxix) . Title:  charter of economic rights and duties of states
Not reporting  3281 (xxix) as cite 12118
However, containing the title 0
Lets take a look

 []


In [19]:
title = 'declaration on the establishment of a new international economic order'
res_id = '3201 (s-vi)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())
print('\n\nSometimes it comes together with the programme of action on the establishment of a new international economic order.\
\nIn the Programme of action snippet we will see that there are resolutions that should account as they cite them both.')

Resolution  3201 (s-vi) . Title:  declaration on the establishment of a new international economic order
Not reporting  3201 (s-vi) as cite 12189
However, containing the title 0
Lets take a look

 []


Sometimes it comes together with the programme of action on the establishment of a new international economic order.
In the Programme of action snippet we will see that there are resolutions that should account as they cite them both.


In [20]:
title = 'programme of action on the establishment of a new international economic order'
res_id = '3202 (s-vi)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
df_cites = df_aux[['res_id2_doc_giving_cite']].drop_duplicates()
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite']))

print('Lets take a look\n\n', df_aux['res_id3_giv'])


print('\n\nAfter a manual inspection, we detect that it comes together with the declaration on the establishment of a new international economic order')

title_combined = 'declaration and the programme of action on the establishment of a new international economic order'

df_aux = df_aux[~df_aux['content_giv'].str.contains(title_combined, case=False)]
print('We filter the ones that do not cite both the declaration and the programm, for a better inspection. \n\nNot containing the title: ', len(df_aux['res_id2_doc_giving_cite']))
print('Lets take a look\n\n', df_aux['res_id3_giv'])


print('\n\nAfter a manual inspection all of them contain typos, all of them should cite both the declaration and the programme of action. Only 32/168 truly contains solely the programm')
df_cites['res_id2_unlet_recv'] = res_id
new_cites = pd.concat([new_cites, df_cites], ignore_index=True)
print('Added to new_cites the cases of Programm of Action resolution: ', res_id)

res_id = '3201 (s-vi)' #declaration on the stablishment of a new economic order
df_cites = df_cites[df_cites['res_id2_doc_giving_cite'] != '32/168']
df_cites['res_id2_unlet_recv'] = res_id
new_cites = pd.concat([new_cites, df_cites], ignore_index=True)
print('Added to new_cites the cases of Declaration on the Stablishment: ', res_id)

Resolution  3202 (s-vi) . Title:  programme of action on the establishment of a new international economic order
Not reporting  3202 (s-vi) as cite 12190
However, containing the title 18
Lets take a look

 4641     3217 (xxix) of 6 november 1974
4712     3251 (xxix) of 4 december 1974
5189      3437 (xxx) of 9 december 1975
5345     3486 (xxx) of 12 december 1975
5624          31/38 of 30 november 1976
5810         31/109 of 16 december 1976
5815         31/111 of 16 december 1976
5944         31/156 of 21 december 1976
6042         31/184 of 21 december 1976
6072         31/202 of 22 december 1976
6659         32/162 of 19 december 1977
6680         32/168 of 19 december 1977
9006         35/67 a of 5 december 1980
9009           35/68 of 5 december 1980
12293         38/25 of 22 november 1983
14874         40/23 of 29 november 1985
14928         40/32 of 29 november 1985
15387        40/111 of 13 december 1985
Name: res_id3_giv, dtype: object


After a manual inspection, we detect th

In [21]:
title = 'scale of assessments for the apportionment of expenses of united nations peacekeeping operations'
res_id = '55/235'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite']))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

Resolution  55/235 . Title:  scale of assessments for the apportionment of expenses of united nations peacekeeping operations
Not reporting  55/235 as cite 5140
However, containing the title 0
Lets take a look

 []


In [23]:
title = 'development and international economic co-operation'
avoid_pattern = (
    r'director-general for development and international economic co-operation'
)
res_id = '3362 (s-vii)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
contains_title = df_aux['content_giv'].str.contains(
    title,
    case=False,
    na=False
)

contains_avoid = df_aux['content_giv'].str.contains(
    avoid_pattern,
    case=False,
    regex=True,
    na=False
)

df_aux = df_aux[contains_title & ~contains_avoid]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

#Some say. decides to put in the agenda ... under the item 'title'. Which does not imply that they are citing the resolution itself.
#Many appeal to the director general.

print('\n\nAfter a inspection. this are the correct ones.')
print('31/109 \n33/155 \n35/56\n37/203')
target_res_ids = ['31/109', '33/155', '35/56', '37/203']

df_cites = df_aux[['res_id2_doc_giving_cite']].drop_duplicates()
df_cites = df_cites[df_cites['res_id2_doc_giving_cite'].isin(target_res_ids)]

assert len(df_cites) == 4
df_cites['res_id2_unlet_recv'] = res_id
new_cites = pd.concat([new_cites, df_cites], ignore_index=True)
print('Added the correct ones to new_cites ')

Resolution  3362 (s-vii) . Title:  development and international economic co-operation
Not reporting  3362 (s-vii) as cite 12108
However, containing the title 10
Lets take a look

 ['31/109 of 16 december 1976' '31/177 of 21 december 1976'
 '33/155 of 20 december 1978' '35/56 of 5 december 1980'
 '37/203 of 20 december 1982' '41/184 of 8 december 1986'
 '42/187 of 11 december 1987' '45/219 of 21 december 1990'
 '47/187 of 22 december 1992' '47/212 b of 6 may 1993']


After a inspection. this are the correct ones.
31/109 
33/155 
35/56
37/203
Added to new_cites the correct ones


In [24]:
title = 'international convention on the elimination of all forms of racial discrimination'
res_id = '2106 (xx)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

Resolution  2106 (xx) . Title:  international convention on the elimination of all forms of racial discrimination
Not reporting  2106 (xx) as cite 13134
However, containing the title 0
Lets take a look

 []


In [25]:
title = 'declaration on principles of international law concerning friendly relations and co-operation among states in accordance with the charter of the united nations'
res_id = '2625 (xxv)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

df_cites = df_aux[['res_id2_doc_giving_cite']].drop_duplicates()
df_cites['res_id2_unlet_recv'] = res_id
new_cites = pd.concat([new_cites, df_cites], ignore_index=True)
print('Added to new_cites.')

Resolution  2625 (xxv) . Title:  declaration on principles of international law concerning friendly relations and co-operation among states in accordance with the charter of the united nations
Not reporting  2625 (xxv) as cite 12774
However, containing the title 37
Lets take a look

 ['2649 (xxv) of 30 november 1970' '2672 c (xxv) of 8 december 1970'
 '2734 (xxv) of 16 december 1970' '2749 (xxv) of 17 december 1970'
 '2916 (xxvii) of 9 november 1972' '3057 (xxviii) of 2 november 1973'
 '3074 (xxviii) of 3 december 1973' '3103 (xxviii) of 12 december 1973'
 '3314 (xxix) of 14 december 1974' '33/73 of 15 december 1978'
 '34/99 of 14 december 1979' '34/140 of 14 december 1979'
 '34/141 of 17 december 1979' '35/163 of 15 december 1980'
 '35/166 of 15 december 1980' '36/76 of 4 december 1981'
 '36/101 of 9 december 1981' '37/10 of 15 november 1982'
 '37/67 of 3 december 1982' '37/109 of 16 december 1982'
 '37/117 of 16 december 1982' '38/7 of 2 november 1983'
 '38/190 of 20 december 1983' '

In [27]:
title = 'transforming our world: the 2030 agenda for sustainable development'
tittle_short = '2030 agenda for sustainable development'
res_id = '70/1'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'])
print('\n\nRevisited: all cite the 70/1 (Mostly also indicated in the footpage)')

df_cites = df_aux[['res_id2_doc_giving_cite']].drop_duplicates()
df_cites['res_id2_unlet_recv'] = res_id
new_cites = pd.concat([new_cites, df_cites], ignore_index=True)
print('Added to new_cites.')

Resolution  70/1 . Title:  transforming our world: the 2030 agenda for sustainable development
Not reporting  70/1 as cite 1009
However, containing the title 14
Lets take a look

 66019        70/75 of 8 december 2015
66416      70/127 of 17 december 2015
66419      70/128 of 17 december 2015
66442      70/132 of 17 december 2015
66495      70/138 of 17 december 2015
66580      70/151 of 17 december 2015
67458    70/248 a of 23 december 2015
69258      71/168 of 19 december 2016
69269      71/169 of 19 december 2016
70989     71/327 of 11 september 2017
72245      72/148 of 19 december 2017
72612      72/196 of 19 december 2017
75199      73/147 of 17 december 2018
75225      73/149 of 17 december 2018
Name: res_id3_giv, dtype: object


Revisited: all cite the 70/1 (Mostly also indicated in the footpage)
Added to new_cites.


In [28]:
title = 'palestine - progress report of the united nations mediator'
res_id = '194 (iii)'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite'].unique()))

print('Lets take a look\n\n', df_aux['res_id3_giv'].unique())

Resolution  194 (iii) . Title:  palestine - progress report of the united nations mediator
Not reporting  194 (iii) as cite 14178
However, containing the title 0
Lets take a look

 []


In [29]:
title = 'united nations convention against transnational organized crime'
res_id = '55/25'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite']))

print('Lets take a look\n\n', df_aux['res_id3_giv'])

Resolution  55/25 . Title:  united nations convention against transnational organized crime
Not reporting  55/25 as cite 5503
However, containing the title 0
Lets take a look

 Series([], Name: res_id3_giv, dtype: object)


In [30]:
title = 'financing of the united nations transition assistance group'
res_id = '43/232'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite']))

print('Lets take a look\n\n', df_aux['res_id3_giv'])

Resolution  43/232 . Title:  financing of the united nations transition assistance group
Not reporting  43/232 as cite 8619
However, containing the title 0
Lets take a look

 Series([], Name: res_id3_giv, dtype: object)


In [38]:
title = 'addis ababa action agenda of the third international conference on financing for development (addis ababa action agenda)'
title_short = 'addis ababa action agenda of the third international conference on financing for development'
title_agenda = 'addis ababa action agenda'

res_id = '69/313'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution', res_id, '. Title:', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

print('Not reporting', res_id, 'as cite:', len(df_aux))

# Scan all title variants
contains_title = (
    df_aux['content_giv'].str.contains(re.escape(title), case=False, na=False) |
    df_aux['content_giv'].str.contains(title_short, case=False, na=False) |
    df_aux['content_giv'].str.contains(title_agenda, case=False, na=False)
)

df_aux = df_aux[contains_title]

print(
    'However, containing one of the title variants:',
    len(df_aux['res_id2_doc_giving_cite'].unique())
)

print('Lets take a look\n\n', df_aux['res_id3_giv'])

print('\n\nAll of them should cite 69/313, the problem is that the title is long, and in many cases the short version is used: addis ababa action agenda')

df_cites = df_aux[['res_id2_doc_giving_cite']].drop_duplicates()
df_cites['res_id2_unlet_recv'] = res_id
new_cites = pd.concat([new_cites, df_cites], ignore_index=True)
print('Added to new_cites.')

Resolution 69/313 . Title: addis ababa action agenda of the third international conference on financing for development (addis ababa action agenda)
Not reporting 69/313 as cite: 1095
However, containing one of the title variants: 59
Lets take a look

 65317     69/318 of 10 september 2015
66234      70/107 of 10 december 2015
66321      70/115 of 14 december 2015
66431      70/129 of 17 december 2015
66442      70/132 of 17 december 2015
66456      70/133 of 17 december 2015
66495      70/138 of 17 december 2015
66668      70/166 of 17 december 2015
67267      70/228 of 23 december 2015
67303      70/235 of 23 december 2015
67458    70/248 a of 23 december 2015
67508        70/248 c of 17 june 2016
67631           70/266 of 8 june 2016
68057      70/302 of 9 september 2016
68064      70/303 of 9 september 2016
68075     70/305 of 13 september 2016
68100       71/1 of 19 september 2016
69012       71/128 of 8 december 2016
69238      71/165 of 19 december 2016
69752      71/222 of 21 de

In [37]:
title = 'review of the efficiency of the administrative and financial functioning of the united nations'
res_id = '41/213'
res_date = pd.to_datetime(
    df2.loc[df2['res_id2_unlet'] == res_id, 'date_p']
).iloc[0]

print('Resolution ', res_id, '. Title: ', title)

#First we get the positive cases: resolution that appear to have cited the res_id
positives = df1[df1['res_id2_unlet_recv'] == res_id]['res_id2_doc_giving_cite'].unique()

#Second we scan the cases where a resolution appears to not cite this res_id
df_aux = df1[df1['res_id2_unlet_recv'] != res_id]

#But filter out the positives cases, the resolutions that state as they have cited the resolution
df_aux = df_aux[~df_aux['res_id2_doc_giving_cite'].isin(positives)]
df_aux = df_aux[df_aux['res_id2_doc_giving_cite'] != res_id]
#Exclude the candidates prior to the resolution, under the principle that resolutions can only cite past resolutions
df_aux['date_p_giv'] = pd.to_datetime(df_aux['date_p_giv'])
df_aux = df_aux[df_aux['date_p_giv'] > res_date]

#We reduce the dataframe, so the scan of the title is not perform in duplicates
df_aux = df_aux[['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv']]
df_aux = df_aux.drop_duplicates(subset=['res_id2_doc_giving_cite', 'res_id3_giv', 'content_giv'])

#We have a total of number of resolution to check if they contain the title and not report the cite
print('Not reporting ', res_id, 'as cite', len(df_aux))

#Last, we scan the title, and print the number of resolutions
df_aux = df_aux[df_aux['content_giv'].str.contains(title, case=False)]
print('However, containing the title', len(df_aux['res_id2_doc_giving_cite']))

print('Lets take a look\n\n', df_aux['res_id3_giv'])

print('There are more resolutions that have the same title. A/RES/60/254 , A/RES/48/218 as an examples. It is better to consider only when specific resolution code is given.')
print('No need to add to new_cites')

Resolution  41/213 . Title:  review of the efficiency of the administrative and financial functioning of the united nations
Not reporting  41/213 as cite 9204
However, containing the title 5
Lets take a look

 26864    48/218 a of 23 december 1993
29869           50/233 of 7 june 1996
35523      54/236 of 23 december 1999
37194    55/220 a of 23 december 2000
43839      59/272 of 23 december 2004
Name: res_id3_giv, dtype: object
There are more resolutions that have the same title. A/RES/60/254 , A/RES/48/218 as an examples. It is better to consider only when specific resolution code is given.
No need to add to new_cites


### Save of the new found connections
Throughout this analysis, the new edges are stored in a dataframe called new_cites. We need to save it as a csv for a further creation of a ground truth of our resolutions.

In [69]:
merged = pd.merge(new_cites, df2, "left", left_on="res_id2_doc_giving_cite", right_on="res_id2")

merged.rename(columns={
    "res_id3": "res_id3_giv",
    "res_id2_unlet": "res_id2_unlet_giv",
    "alt_id_dic": "alt_id_dic_giv",
    "session_type": "session_type_giv",
    "session_reg": "session_reg_giv",
    "session_sp": "session_sp_giv",
    "session_es": "session_es_giv",
    "resn": "resn_giv",
    "res_letter": "res_letter_giv",
    "date_p": "date_p_giv",
    "date_c": "date_c_giv",
    "location": "location_giv",
    "topic": "topic_giv",
    "filename": "filename_giv",
    "content": "content_giv"
}, inplace=True)

# Drop record, draft, n_inc_cit
merged = merged.drop(columns=["res_id2", "record", "draft", "n_inc_cit"], errors="ignore")

# Add the missing columns
merged["cited_res"] = ""
merged["cited_res_conv"] = ""
merged["n_outg_cit"] = ""


###### Now with the receiving fields
merged = pd.merge(merged, df2, "left", left_on="res_id2_unlet_recv", right_on="res_id2_unlet")

merged.rename(columns={
    "res_id2": "res_id2_doc_receiv_cite",
    "res_id3": "res_id3_recv",
    "alt_id_dic": "alt_id_dic_recv",
    "session_type": "session_type_recv",
    "session_reg": "session_reg_recv",
    "session_sp": "session_sp_recv",
    "session_es": "session_es_recv",
    "resn": "resn_recv",
    "res_letter": "res_letter_recv",
    "date_p": "date_p_recv",
    "date_c": "date_c_recv",
    "location": "location_recv",
    "topic": "topic_recv",
    "filename": "filename_recv",
    "content": "content_recv"
}, inplace=True)

merged = merged.drop(columns=["res_id2_unlet", "record", "draft", "n_inc_cit"], errors="ignore")

assert set(df1.columns) == set(merged.columns)


Index(['res_id2_doc_giving_cite', 'res_id2_unlet_recv', 'res_id3_giv',
       'res_id2_unlet_giv', 'alt_id_dic_giv', 'session_type_giv',
       'session_reg_giv', 'session_sp_giv', 'session_es_giv', 'resn_giv',
       'res_letter_giv', 'date_p_giv', 'date_c_giv', 'filename_giv',
       'content_giv', 'location_giv', 'topic_giv', 'cited_res',
       'cited_res_conv', 'n_outg_cit', 'res_id2', 'res_id3', 'res_id2_unlet',
       'alt_id_dic', 'session_type', 'session_reg', 'session_sp', 'session_es',
       'resn', 'res_letter', 'date_p', 'date_c', 'filename', 'content',
       'location', 'record', 'draft', 'topic', 'n_inc_cit'],
      dtype='object')
(1405, 39)
Index(['res_id2_doc_giving_cite', 'res_id2_unlet_recv', 'res_id3_giv',
       'res_id2_unlet_giv', 'alt_id_dic_giv', 'session_type_giv',
       'session_reg_giv', 'session_sp_giv', 'session_es_giv', 'resn_giv',
       'res_letter_giv', 'date_p_giv', 'date_c_giv', 'filename_giv',
       'content_giv', 'location_giv', 'topic_giv', '

In [70]:
all = pd.concat([df1, merged], ignore_index=True)
all = all.drop_duplicates()

all.to_csv(CITATIONS_CSV, index=False)
print("New records saved!")

Save new records!
